In [1]:
import mlflow, psycopg2
print("mlflow:", mlflow.__version__)

mlflow: 2.16.2


In [2]:
#!pip install psycopg2-binary mlflow==2.16.2import os -> se hace desde la creación del contenedor
import os
import pandas as pd
import sqlalchemy as sa

engine = sa.create_engine(os.environ["DATA_DB_CONN"])
df = pd.read_csv("/home/jovyan/data/penguins.csv")

print(f"Filas en el CSV: {len(df)}")
df.head()

Filas en el CSV: 344


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [3]:
with engine.begin() as conn:
    conn.execute(sa.text("TRUNCATE TABLE raw.penguins_raw RESTART IDENTITY"))

crudos = df.astype(str)
crudos.to_sql("penguins_raw", engine, schema="raw",
              if_exists="append", index=False)

total = pd.read_sql("SELECT COUNT(*) AS n FROM raw.penguins_raw", engine)
print(f"Filas en raw: {total['n'][0]}")

Filas en raw: 344


In [4]:
limpio = df.copy()
inicial = len(limpio)

limpio = limpio.rename(columns={
    "bill_length_mm": "culmen_length_mm",
    "bill_depth_mm": "culmen_depth_mm",
})

limpio = limpio.dropna()
sin_nulos = len(limpio)

limpio["sex"] = limpio["sex"].str.upper().str.strip()
limpio = limpio[limpio["sex"].isin(["MALE", "FEMALE"])]
sexo_valido = len(limpio)

limpio = limpio.drop_duplicates()
final = len(limpio)

limpio = limpio[["species", "island", "culmen_length_mm", "culmen_depth_mm",
                 "flipper_length_mm", "body_mass_g", "sex"]]

with engine.begin() as conn:
    conn.execute(sa.text("TRUNCATE TABLE clean.penguins_clean RESTART IDENTITY"))

limpio.to_sql("penguins_clean", engine, schema="clean",
              if_exists="append", index=False)

print(f"raw: {inicial} -> sin nulos: {sin_nulos} -> sexo valido: {sexo_valido} -> sin duplicados: {final}")
print(f"\nDistribucion de especies:\n{limpio['species'].value_counts()}")

raw: 344 -> sin nulos: 333 -> sexo valido: 333 -> sin duplicados: 333

Distribucion de especies:
species
Adelie       146
Gentoo       119
Chinstrap     68
Name: count, dtype: int64
